Пусть теперь контроллер принимает лямбда-функцию ускорения - мы уходим от идеи выйти на равномерное движение с желаемой скорости

Ограничения силы двигателя реализуем ограничением ускорения, как и ранее: $| \ddot{x}(t)| \le a_{max} \forall t$

Ограничение скорости возьмем стандартное из литературы 120 км/ч = 33.3 м/с

Сделаем возможность альфа-агенту быть не только во главе роя. В этом случае всем после него надо будет ориентироваться на предыдущего агента.

Принимаем, что до начала полета агенты знают, где альфа. Тогда те, кто летит следом за $\alpha$, сохраняют старый вариант ускорения

$$ \forall i<\alpha: \quad l_i = s_0 + \tau \cdot v_i, \quad k_i = \frac{m a_{max}}{l_i}, \quad
b_i  = \max{(\frac{m}{\tau}, \sqrt{k_i m})}, \\ a_i = \frac{1}{m} (k_i \cdot (x_{i+1} - x_i - l_i) + b_i \cdot (v_{i+1} - v_i))
$$

На текущий момент принимаем, что при обмене информацией получаем точную скорость с борта

<!--
Для них остаются прежними соотношения в фильтре Калмана для уточнения скоростей:
$$
X_{k+1} = F X_k + B a_k + V_k, \\
X_k = \begin{bmatrix} x_{i+1} - x_{i} \\ v_i \\ v_{i+1} \end{bmatrix}, \quad F = \begin{bmatrix}
1 & -\Delta t & \Delta t \\
0 & 1 & 0 \\
0 & 0 & 1
\end{bmatrix}, \quad
B = \begin{bmatrix}
-\tfrac{1}{2}\Delta t^2 \\
\Delta t\\
0
\end{bmatrix}, \quad
V_k \sim \mathcal{N}(0, Q) \\
Q = q_v\, B_v B_v^\top + q_{v^+}\, B_{v^+} B_{v^+}^\top = \begin{bmatrix}
\frac{1}{4}(q_v + q_{v^+})\Delta t^4 & -\frac{1}{2}q_v \Delta t^3 & \frac{1}{2}q_{v^+} \Delta t^3 \\
-\frac{1}{2}q_v \Delta t^3 & q_v \Delta t^2 & 0 \\
\frac{1}{2}q_{v^+} \Delta t^3 & 0 & q_{v^+} \Delta t^2
\end{bmatrix}, \quad B_v = B,
B_{v^+} = \begin{bmatrix} \tfrac{1}{2}\Delta t^2 \\ 0 \\ \Delta t \end{bmatrix}
$$

А летящих впереди подталкивают сзади
$$
\forall i > \alpha: \quad a_i = \frac{1}{m} (k_i \cdot (l_i - x_{i} + x_{i-1}) + b_i \cdot (v_{i-1} - v_i))
$$
Тогда матрицы в фильтрации будет выглядеть чуть иначе:
$$
X_k = \begin{bmatrix} x_{i} - x_{i-1} \\ v_i \\ v_{i-1} \end{bmatrix}, \quad
F = \begin{bmatrix}
1 & \Delta t & -\Delta t \\
0 & 1 & 0 \\
0 & 0 & 1
\end{bmatrix}, \quad
B = \begin{bmatrix}
\frac{1}{2}\Delta t^2 \\
\Delta t\\
0
\end{bmatrix}, \quad
V_k \sim \mathcal{N}(0, Q) \\
Q = q_v\, B_v B_v^\top + q_{v^+}\, B_{v^-} B_{v^-}^\top, B_v = B,
B_{v^-} = \begin{bmatrix} -\frac{1}{2}\Delta t^2 \\ 0 \\ \Delta t \end{bmatrix}
$$

update2: упругость $k$ и дампинг $b$ считаем с поправкой на расстояние от альфа агента, чтобы дальние агенты острее реагировали - до них изменения доходят дольше

update3: чуть изменяем подсчет длины виртуальной пружины для агентов, которые летят перед альфой, в момент когда альфа летит назад (от них) - добавляем небольшой буфер.

-->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from typing import Tuple, List, Callable, Literal
import itertools
import math
from tqdm import tqdm

In [ ]:
CycType = Literal["Trajectories", "Velocities", "Accelerations"]
g = 9.81
v_max = 100/3

In [ ]:
def cyclogram(traj, times, alpha_idx=0, ax=None, relative=True, limit=None, idx=None, type_data: CycType ='Trajectories'):
    """
    Визуализируем траекторию, скорость или ускорение (для двух последних - только альфу)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    ticks_fontsize=16
    labels_fontsize=16
    legend_fontsize=16

    traj = np.asarray(traj)
    times = np.asarray(times)
    n_steps = len(times)

    # Обрезаем, если смотрим конечный участок
    if limit is not None and limit > 0:
        start = max(0, n_steps - limit)
        plot_traj = traj[start:]
        plot_times = times[start:]
    else:
        plot_traj = traj
        plot_times = times

    N = plot_traj.shape[1]

    # сдвигаем наблюдения если смотрим относительно альфы
    if relative:
        alpha_col = plot_traj[:, alpha_idx][:, np.newaxis]
        positions = plot_traj - alpha_col
    else:
        positions = plot_traj

    # Цвета траекторий: черный у альфы, у остальных разноцветные
    cmap = plt.cm.tab20
    forward = [cmap(i % 20) for i in range(alpha_idx)][::-1]
    backward = [cmap(i % 20) for i in range(N - alpha_idx - 1)]
    colors = forward + ['black'] + backward

    # Отрисовка
    if type_data == "Trajectories":
        for i in range(N):
            label = "Alpha" if i == alpha_idx else f"{i+1}"
            # Жирность линии для альфы
            ms = 3 if i == alpha_idx else 1
            ax.plot(plot_times, positions[:, i],
                label=label, marker='o', markersize=ms,
                color=colors[i],#  if i != alpha_idx else 'black'
                linewidth=1.0)
    else:
        ax.plot(plot_times, positions[:, alpha_idx], color='black')

    # Оси
    ax.set_xlabel('$t$ [s]', loc='right', fontsize=labels_fontsize)

    y_label = "$x$ [m]" if type_data=="Trajectories" else "$v$ [m/s]" if type_data=="Velocities" else "$a$ [$m/s^2$]"
    ax.set_ylabel(y_label, loc='top', fontsize=labels_fontsize, rotation=0)
    ax.yaxis.set_label_coords(x=-0.1, y=0.94, transform=ax.transAxes)

    ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize)

    # Заголовок
    title = f'Cyclogram {idx if idx is not None else ""}: Agents {type_data}'
    if relative:
        title += ", relative to Alpha"
    if limit is not None:
        title += f", last {limit} steps"
    ax.set_title(title)
    ax.grid(True, linestyle='--', alpha=0.6)

    # Легенда за пределами картинки
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], loc='upper left', fontsize=legend_fontsize,
              bbox_to_anchor=(1.05, 1), borderaxespad=0)

    # начало координат без смещения
    ax.margins(x=0, y=0)
    return ax

Это класс легаси, в настоящий момент Калман нам не нужен

In [ ]:
class RelativeDEKF3D:
    def __init__(self, s0=2.0, dt=0.2, q_v=0.1, q_vp=0.1, R_d=1e-4, sigma=1e-6, rev=False):
        # летим за альфой => [d_i=x_{i+1}-x_i, v_i, v_{i-1}], перед альфой => [d_i=x_i - x_{i-1}, v_i, v_{i+1}]

        # Вектор состояния
        self.x_hat = np.array([s0, 0.0, 0.0])
        # Ошибка
        self.P = np.eye(3) * sigma
        self.dt = dt
        # Шум измерения
        self.R_d = R_d
        # Коэффициент авторегрессии
        self.F = np.array([
            [1, dt,  -dt],
            [0,  1,   0],
            [0,  0,   1]
        ]) if rev else np.array([
            [1, -dt,  dt],
            [0,  1,   0],
            [0,  0,   1]
        ])

        # Шум истинного состояния
        self.B = np.array([0.5*dt**2, dt, 0.0]) if rev else np.array([-0.5*dt**2, dt, 0.0])
        B_v  = np.copy(self.B)
        B_vp = np.array([-0.5*dt**2, 0.0, dt]) if rev else np.array([ 0.5*dt**2, 0.0, dt])
        self.Q = q_v * np.outer(B_v, B_v) + q_vp * np.outer(B_vp, B_vp)

        # Для подсчета kalman gain
        self.H = np.array([[1.0, 0.0, 0.0]])

    def predict(self, a_cmd):
        self.x_hat = self.F @ self.x_hat + self.B * a_cmd
        self.P = self.F @ self.P @ self.F.T +  self.Q

    def update(self, d_meas):
        if d_meas is None:
            return
        y = d_meas - self.H @ self.x_hat
        S = self.H @ self.P @ self.H.T + self.R_d
        K = self.P @ self.H.T / S
        self.x_hat += K.flatten() * y
        self.P = (np.eye(3) - K @ self.H) @ self.P

    def estimate(self, a_cmd, d_meas):
        # Предсказываем
        self.predict(a_cmd)
        # Корректируем
        self.update(d_meas)
        return self.x_hat.copy()

In [ ]:
class Swarm:
    """
    Рой, хранящий историю наблюдений
    """
    def __init__(self, N=5, s0=2.0, dt=0.2, alpha_idx = None, a_max=g, v_max = v_max):
        self.N = N
        self.dt = dt
        self.t = 0.0
        self.s0 = s0
        self.alpha_idx = alpha_idx if alpha_idx is not None else N-1
        self.a_max = a_max
        self.v_max = v_max

        # Координаты агентов
        self.x = np.arange(0, N * s0, s0)
        # Скорости
        self.v = np.zeros(N)
        # Ускорения
        self.a = np.zeros(N)
        # Метки времени
        self.timestamps = [self.t]
        # История состояний
        self.trajectory = [self.x.copy()]
        self.velocities = [self.v.copy()]
        self.accelerations = [self.a.copy()]

        # На текущий момент не применяется
        self.ekfs = [RelativeDEKF3D(s0=self.s0, dt=dt, rev=i > self.alpha_idx) if i != self.alpha_idx else None for i in range(N)]

    def step(self, acc_cmd):
        """
        Тик времени - обновление состояний
        """
        acc_cmd = np.clip(acc_cmd, -self.a_max, self.a_max)
        self.x += self.v * self.dt + 0.5 * acc_cmd * self.dt**2

        v_new = np.clip(self.v + acc_cmd * self.dt, -self.v_max, self.v_max)
        self.v = v_new
        self.a = acc_cmd.copy()
        self.t += self.dt
        self.timestamps.append(self.t)

        self.trajectory.append(self.x.copy())
        self.velocities.append(self.v.copy())
        self.accelerations.append(self.a.copy())

In [ ]:
class SMDController:
    """
    Вся логика получения ускорений здесь.
    Рой получает только итоговые ускорения
    """
    def __init__(self, ddot_x, m=1676.0, a_max=3.7, s0=2.0, dt=0.2, N=10, tau_lag = 0.5):
        # Масса
        self.m = m
        # Максимальное ускорение
        self.a_max = a_max
        # Минимальная безопасная дистанция
        self.s0 = s0
        # Тик времени
        self.dt = dt
        # Функция ускорения для альфы
        self.ddot_x = ddot_x
        # Количество агентов
        self.N = N
        # Время лага (реагирования)
        self.tau_lag = tau_lag

    def control_step(self, swarm):
        N = self.N
        acc = np.zeros(N)
        t = swarm.t
        dt = swarm.dt
        # Альфа агент летит по своей траектории
        acc[swarm.alpha_idx] = self.ddot_x(t+dt)

        for i in range(N):
            # Скорость самого агента и скорость агента за/перед ним
            v_i_hat, v_ip1_hat = 0,0

            # Получаем скорости
            if i == swarm.alpha_idx:
                continue
            if i > swarm.alpha_idx:
                d_meas = swarm.x[i] - swarm.x[i-1]
                v_i_hat, v_ip1_hat = swarm.v[i], swarm.v[i-1]
            else:
                d_meas = swarm.x[i+1] - swarm.x[i]
                v_i_hat, v_ip1_hat = swarm.v[i], swarm.v[i+1]

            # Безопасная дистанция для рассчета длины пружинки
            min_gap = 0.2 * self.s0

            # Здесь я пытался по-умному рассчитать исходную длину пружинки, но лучше всего подошло просто удвоение максимального расстояния
            # safety_mult = 1.0 + 0.5 * self.N * (self.a_max * (self.tau_lag)**2 / self.s0)
            ell_i = self.s0 * 2

            if i > swarm.alpha_idx:
                # Буфер только при движении назад, когда есть риск столкновения с альфой
                buffer = self.tau_lag * max(0, -v_i_hat)
                ell_i += buffer
            else:
                ell_i += self.tau_lag * v_i_hat
            # Длина пружины не должна стать отрицательной
            ell_i = max(ell_i, min_gap)

            k_i = self.m * self.a_max / ell_i
            b_i = max(self.m / self.tau_lag, np.sqrt( k_i * self.m))
            # Коэффициент для усиления реакции, чтобы дроны не налетали на передних
            if i > swarm.alpha_idx:
              k_i *= (1+ 0.05 * np.abs(i-swarm.alpha_idx))
              b_i *= (1+ 0.05 * np.abs(i-swarm.alpha_idx))
            # Сжатие пружинки
            deltaX = ell_i - d_meas if i > swarm.alpha_idx else d_meas - ell_i
            # Относительная скорость
            deltaV = v_ip1_hat - v_i_hat
            acc[i] = (k_i * deltaX + b_i * deltaV) / self.m
        return acc

симуляция $\sim 20$ агентов в течение $\sim 2$ часов - 1 минута

Чтобы были маневры, функция ускорения альфы $\displaystyle a(t) = \sum_{i=1}^K A_i \sin{\omega_i t} + B_i \cos{\omega_i t}$. Поскольку аппаратно ускорение ограничено $a_{max}$, требуем $\displaystyle \sum_{i=1}^K \sqrt{A_i^2 + B_i^2} \leq a_{max}$

И также добавляем класс для моделирования резкой последовательности ускорений-замедлений вида
$$
\ddot{x} = \begin{cases} a_{max}, t \in [\tau, 3\tau) \\ -a_{max}, t \in [0, \tau), [3 \tau, 4 \tau) \end{cases}
$$

In [ ]:
class BangBangTrajectory:
    """
    Класс для резкого управления, периодически включаем противоположные максимальные ускорения
    """
    def __init__(self, a_max=4.0, tau=2.0):
        self.a_max = a_max
        self.tau = tau
        self.phases = [
            (0.0, tau, a_max),
            (tau, 3*tau, -a_max),
            (3*tau, 4*tau, a_max),
        ]
        self.T = 4 * tau

    def __call__(self, t):
        t_mod = t % self.T
        for t0, t1, a in self.phases:
            if t0 <= t_mod < t1:
                return a
        return self.phases[-1][2]

In [ ]:
def run_simulation(N, alpha_idx, s0, dt, steps, a_max, m, ddotx_func):
    """
    Функция для полноценного запуска одной симуляции
    """
    maxd = N * s0 * 5
    swarm = Swarm(N=N, s0=s0, dt=dt, alpha_idx=alpha_idx)
    ctrl = SMDController(ddot_x=ddotx_func, m=m, a_max=a_max, s0=s0, dt=dt, N=N)
    for _ in tqdm(range(steps)):
        acc = ctrl.control_step(swarm)
        swarm.step(acc)
        # После шага алгоритма проверяем условия поддержания роя
        broke = False
        for i in range(swarm.N-1):
            # Опасное сближение
            if swarm.x[i] >= swarm.x[i+1] or swarm.x[i+1] - swarm.x[i] < s0 * 0.1:
              print(f'\n{i} x {i+1}: BREAK')
              broke = True
              break
        if broke:
            break
        # Рой разлетелся
        if np.abs(np.max(swarm.x) - np.min(swarm.x)) > maxd:
          print('\nflew away')
          break
    return swarm

In [ ]:
flights_params = [
    # N, alpha_idx, s0, dt, steps, a_max, m, ddotx_func
    (6, None, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=3)), # <-
    (6, 0, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=3)),
    (6, None, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=4)), # <-
    (6, 0, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=4)), # <-
    (6, None, 20.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=5)),
    (6, None, 18.6, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=5)),
    (6, None, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=5)), # <-
    (6, 0, 10.0, 0.1, 8000, g*7/8, 5, BangBangTrajectory(a_max=g*7/8, tau=5)),
    (6, 0, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=5)),
    (10, None, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=5)),
    (15, 5, 10.0, 0.1, 8000, g, 5, BangBangTrajectory(a_max=g, tau=3)),
]

flights_results = []

for i, (N, alpha_idx, s0, dt, steps, a_max, m, ddotx_func) in enumerate(flights_params):
    flights_results.append(run_simulation(N, alpha_idx, s0, dt, steps, a_max, m, ddotx_func))

100%|██████████| 8000/8000 [00:01<00:00, 4032.25it/s]


Визуализация:

In [ ]:
fig, axes = plt.subplots(len(flights_params), 4, figsize=(2.5* max(np.ceil(1.2 * 17), 14 ), len(flights_params) * max(np.ceil(0.7* 15),5)))
for i, swrm in enumerate(flights_results):
    # Весь маршрут, абсолютный
    cyclogram(swrm.trajectory, swrm.timestamps, swrm.alpha_idx, axes[0] if len(flights_params) == 1 else axes[i][0], relative=False, idx=i)
    # Абсолютные расстояния, последние 100 секунд маршрута
    cyclogram(swrm.trajectory, swrm.timestamps, swrm.alpha_idx, axes[1] if len(flights_params) == 1 else axes[i][1], relative=False, limit=100, idx=i)
    # Весь маршрут, относительный
    cyclogram(swrm.trajectory, swrm.timestamps, swrm.alpha_idx, axes[2] if len(flights_params) == 1 else axes[i][2], relative=True, idx=i)
    # Относительные расстояния, последние 100 секунд маршрута
    cyclogram(swrm.trajectory, swrm.timestamps, swrm.alpha_idx, axes[3] if len(flights_params) == 1 else axes[i][3], relative=True, limit=100, idx=i)
    # cyclogram(swrm.velocities, swrm.timestamps, swrm.alpha_idx, axes[4] if len(flights_params) == 1 else axes[i][4], relative=False, idx=i, type_data="Velocities")
    # cyclogram(swrm.accelerations, swrm.timestamps, swrm.alpha_idx, axes[5] if len(flights_params) == 1 else axes[i][5], relative=False, idx=i, type_data="Accelerations")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

Другие менее сложные тесты, которые можно вставить в flights_params выше

In [ ]:
    # (22, 20, 10.0, 0.2, 4000, 5, 10, lambda T: 1*np.sin(T) - 1*np.cos(2*(T)) + 2 * np.sin(3*T) + 2*np.sin(4*(T))),
    # (22, 21, 10.0, 0.2, 4000, 3.7, 10, lambda T: 1*np.sin(T) - 1*np.sin(2*(T))),
    # (22, 5, 10.0, 0.2, 4000, 3.7, 10, lambda T: 1*np.sin(T) - 1*np.sin(2*(T))),
    # (22, 5, 10.0, 0.2, 4000, np.sqrt(5), 10, lambda T: -2*np.sin(10*T) + 1* np.cos(1*T)),
    # (22, 1, 10.0, 0.2, 4000, np.sqrt(5), 10, lambda T: -2*np.sin(10*T) + 1* np.cos(1*T)),
    # (21, 20, 10.0, 0.2, 4000, 5, 10, BangBangTrajectory(a_max=3.7, tau=2)),
    # (21, 20, 10.0, 0.2, 4000, 3.7, 10, BangBangTrajectory(a_max=3.7, tau=3)),
    # (21, 4, 10.0, 0.2, 4000, 3.7, 10, BangBangTrajectory(a_max=3.7, tau=3)),
    # (21, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=2)),
    # (30, 4, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=2)),
    # (30, 4, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1.5)),
    # (21, 2, 10.0, 0.2, 4000, 3.7, 10, BangBangTrajectory(a_max=3.7, tau=2)),
    # (21, 1, 10.0, 0.2, 4000, 3.7, 10, BangBangTrajectory(a_max=3.7, tau=2)),
    # (21, 1, 10.0, 0.2, 4000, 3.7, 10, BangBangTrajectory(a_max=3.7, tau=1.5)),
    # (21, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=3)), #
    # (21, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=2)),
    # (21, 20, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1.5)),
    #  (20, None, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1.5)), #
    # (20, None, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=2)), #
    # (20, None, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=3)), #
    # (20, None, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=4)),
    # (20, None, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1)),
    # (20, None, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1)),
    # (5, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1)),
    #  (21, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1.5)),
    # (18, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1.5)),
    # (19, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1.5)),
    # (20, 0, 10.0, 0.2, 4000, 6, 5, BangBangTrajectory(a_max=6, tau=1.5)),